# 07 - Cluster evaluation

Objetivo: evaluar separacion estadistica, tamanos, ruido y estabilidad de una configuracion candidata.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import polars as pl
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features, transform_features, run_clustering, evaluate_labels

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
features = build_customer_features(load_transactions(ROOT / "transacciones_sinteticas.csv"))
matrix, _ = transform_features(features, scaler="robust")
configurations = [("kmeans", k) for k in (3, 4, 5, 6)] + [("agglomerative", k) for k in (3, 4, 5, 6)]
rows = []
for algorithm, k in configurations:
    labels = run_clustering(matrix, algorithm, n_clusters=k)
    metrics = evaluate_labels(matrix, labels)
    counts = np.bincount(labels[labels >= 0])
    rows.append({"algorithm": algorithm, "k": k, **metrics, "smallest_cluster": int(counts.min()), "largest_cluster": int(counts.max())})
evaluation = pl.DataFrame(rows)
evaluation.write_csv(ROOT / "results" / "metrics" / "cluster_evaluation.csv")
evaluation

In [ ]:
candidate = evaluation.sort(["silhouette", "davies_bouldin"], descending=[True, False]).row(0, named=True)
print("Candidata estadistica, pendiente de validar interpretabilidad:", candidate)

## Conclusion
Una metrica favorable no basta para afirmar que existen segmentos utiles. La decision final se contrasta con perfiles y estabilidad.